# LCESA Pipeline — Colab Notebook
# Low-Curvature Endogenous Standpoint Attractor

Run the full LCESA curvature validation pipeline on Google Colab (free T4 GPU).

## Setup
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells in order
3. For Llama models: paste your HuggingFace token when prompted

## Models
| Model | VRAM (fp16) | VRAM (4-bit) | Colab Free |
|-------|-------------|-------------|------------|
| GPT-2 | ~0.1 GB | N/A | ✅ |
| Llama-2-7b | ~4.3 GB | ~1.1 GB | ✅ (use 4-bit) |
| Llama-2-13b | ~8.4 GB | ~2.1 GB | ✅ (use 4-bit) |

In [ ]:
# Install dependencies
!pip install transformer_lens bitsandbytes accelerate scipy scikit-learn pandas matplotlib

In [ ]:
# Clone repository and mount Google Drive
import os
from pathlib import Path

REPO_DIR = Path('/content/Low-Curvature-Endogenous-Standpoint-Attractor')
DRIVE_RESULTS = Path('/content/drive/MyDrive/lcesa_results')

# Mount Google Drive for persistent storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Results will be saved to: {DRIVE_RESULTS}")
except Exception as e:
    print(f"Drive mount failed (results won't persist): {e}")
    DRIVE_RESULTS = Path('/content/lcesa_results')
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

# Clone repo if not already present
if not REPO_DIR.exists():
    !git clone https://github.com/zhangze1007/Low-Curvature-Endogenous-Standpoint-Attractor.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {list(Path('.').iterdir())}")

In [ ]:
# HuggingFace authentication (required for Llama models)
# Get your token from: https://huggingface.co/settings/tokens
# Also accept Llama-2 license at: https://huggingface.co/meta-llama/Llama-2-7b-chat-hf

HF_TOKEN = None  # Set your token here, or use Colab secrets

if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        print("HF token loaded from Colab secrets")
    except Exception:
        print("No HF token found. Set HF_TOKEN variable or add 'HF_TOKEN' to Colab secrets.")
        print("Needed only for Llama models, not for GPT-2.")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("HuggingFace login successful")

In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    print(f"VRAM: {gpu.total_mem / 1e9:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
    print("Pipeline will run on CPU (very slow for Llama models).")

In [ ]:
# Configure model selection
# Change MODEL_NAME to run different models
MODEL_NAME = "gpt2"  # Options: "gpt2", "llama-7b", "llama-13b"
FORCE_4BIT = False   # Set True for Llama models on free-tier Colab

# For Llama on Colab free tier, use:
# MODEL_NAME = "llama-7b"
# FORCE_4BIT = True

import sys
sys.path.insert(0, '.')
from experiments.config import MODELS, get_model_config

config = get_model_config(MODEL_NAME, force_4bit=FORCE_4BIT)
print(f"Model: {config.name} ({config.hf_name})")
print(f"Layers: {config.n_layers}, Heads: {config.n_heads}, d_model: {config.d_model}")
print(f"4-bit: {config.load_in_4bit}")
print(f"Available models: {list(MODELS.keys())}")

In [ ]:
# Step 1: Generate stimuli (shared across models)
from experiments.stimuli.generate import generate_all_stimuli
from experiments.config import DATA_DIR

stimuli_path = DATA_DIR / 'stimuli.json'
if not stimuli_path.exists():
    print("Generating stimuli...")
    generate_all_stimuli()
else:
    print(f"Stimuli already exist at {stimuli_path}")
    import json
    with open(stimuli_path) as f:
        data = json.load(f)
    for s in sorted(data.keys()):
        n = len(data[s].get('grouping', [])) + len(data[s].get('test', []))
        print(f"  {s}: {n} conversations")

In [ ]:
# Step 2: Extract activations (GPU-bound step)
from experiments.extraction.extract import extract_all
from experiments.config import CACHE_DIR

activations_path = CACHE_DIR / MODEL_NAME / 'activations.npz'
if not activations_path.exists():
    print(f"Extracting activations for {MODEL_NAME}...")
    print("This is the main GPU-bound step. Expect ~5-15 min for GPT-2, ~30-60 min for Llama-7b.")
    extract_all(stimuli_path, MODEL_NAME)
else:
    import os
    size_gb = os.path.getsize(activations_path) / 1e9
    print(f"Activations already exist: {activations_path} ({size_gb:.1f} GB)")

In [ ]:
# Step 3: Head grouping
from experiments.grouping.head_grouping import run_head_grouping
from experiments.config import DATA_DIR

gamma_path = DATA_DIR / f'{MODEL_NAME}_grouping.npz'
if not gamma_path.exists():
    print(f"Running head grouping for {MODEL_NAME}...")
    run_head_grouping(MODEL_NAME, activations_path)
else:
    import numpy as np
    gamma = np.load(gamma_path)['gamma']
    from experiments.config import LAYER_NAMES
    print(f"Grouping exists. Head assignments:")
    for idx, name in enumerate(LAYER_NAMES):
        print(f"  {name}: {int(np.sum(gamma == idx))} heads")

In [ ]:
# Step 4: Curvature computation
from experiments.curvature.compute import run_curvature_computation
from experiments.config import RESULTS_DIR

curvature_path = RESULTS_DIR / f'{MODEL_NAME}_curvature.csv'
if not curvature_path.exists():
    print(f"Computing curvature for {MODEL_NAME}...")
    run_curvature_computation(MODEL_NAME, activations_path, gamma_path)
else:
    import pandas as pd
    df = pd.read_csv(curvature_path)
    print(f"Curvature exists: {len(df)} rows, {df['scenario'].nunique()} scenarios")

In [ ]:
# Step 5: Baselines (probing, CKA, entropy, permutation)
from experiments.baselines.linear_probing import run_linear_probing
from experiments.baselines.cka import run_cka
from experiments.baselines.attention_entropy import run_attention_entropy
from experiments.baselines.permutation import run_permutation_test

print("Running baselines...")
print("  Linear probing...")
run_linear_probing(MODEL_NAME, activations_path)
print("  CKA...")
run_cka(MODEL_NAME, activations_path)
print("  Attention entropy...")
run_attention_entropy(MODEL_NAME, activations_path)
print("  Permutation test...")
run_permutation_test(MODEL_NAME, activations_path, gamma_path)
print("Baselines complete.")

In [ ]:
# Step 6: Hypothesis tests (H1-H3, H6)
from experiments.stats.hypothesis_tests import run_all_tests
import json

results = run_all_tests(MODEL_NAME)

# Print enriched summary
print("\n" + "="*60)
print(f"Hypothesis Test Summary — {MODEL_NAME}")
print("="*60)
for h_name, h_result in results.items():
    if h_name == "summary":
        continue
    if isinstance(h_result, dict):
        if "passed" in h_result:
            status = "PASSED" if h_result["passed"] else "FAILED"
            d = h_result.get("cohens_d", h_result.get("curvature_cohens_d", "N/A"))
            p = h_result.get("p_value", None)
            if isinstance(p, float):
                print(f"  {h_name}: {status} (d={d:.3f}, p={p:.2e})")
            else:
                print(f"  {h_name}: {status}")
        elif h_result.get("passed") is None and h_result.get("skipped"):
            print(f"  {h_name}: SKIPPED ({h_result.get('reason', 'no data')})")
print("="*60)

In [ ]:
# Step 7: Visualizations
from experiments.visualization.plots import plot_curvature_heatmap, plot_curvature_boxplots
import pandas as pd

df = pd.read_csv(curvature_path)
print("Generating visualizations...")
plot_curvature_heatmap(df, MODEL_NAME)
plot_curvature_boxplots(df, MODEL_NAME)
print("Figures saved to results/figures/")

In [ ]:
# Copy results to Google Drive for persistence
import shutil
from pathlib import Path

results_dir = Path('results')
drive_dir = DRIVE_RESULTS / MODEL_NAME
drive_dir.mkdir(parents=True, exist_ok=True)

copied = []
# Copy result CSVs and JSONs
for f in results_dir.glob(f'{MODEL_NAME}_*'):
    shutil.copy2(f, drive_dir / f.name)
    copied.append(f.name)

# Copy figures
fig_dir = results_dir / 'figures'
if fig_dir.exists():
    for f in fig_dir.glob(f'{MODEL_NAME}_*'):
        shutil.copy2(f, drive_dir / f.name)
        copied.append(f.name)

# Copy activation cache for ablation studies (~5-20 GB)
cache_file = Path('cache') / MODEL_NAME / 'activations.npz'
if cache_file.exists():
    cache_dest = drive_dir / 'activations.npz'
    shutil.copy2(cache_file, cache_dest)
    copied.append(f'activations.npz ({cache_file.stat().st_size / 1e9:.1f} GB)')

print(f"\nCopied {len(copied)} files to Google Drive: {drive_dir}")
for f in copied:
    print(f"  {f}")
print(f"\nThese files persist across Colab sessions.")

## Results Summary

All results are saved to:
- **Local:** `results/{MODEL_NAME}_*.csv`, `results/{MODEL_NAME}_*.json`
- **Google Drive:** `lcesa_results/{MODEL_NAME}/` (persists across sessions)

### Next Steps
1. **Run on Llama-7b:** Change `MODEL_NAME` to `"llama-7b"` and `FORCE_4BIT` to `True`
2. **Ablation studies:** Run the ablation notebook cell below
3. **Cross-model comparison:** Run `python -m experiments.cross_model gpt2 llama-7b` locally
4. **Download results:** Files are in your Google Drive under `lcesa_results/`

In [ ]:
# Optional: Run ablation study (takes ~10-30 min)
# Uncomment and run after the main pipeline completes

# from experiments.ablation import run_ablation_study
# print(f"Running ablation study for {MODEL_NAME}...")
# ablation_df = run_ablation_study(MODEL_NAME, activations_path, gamma_path)
# print(f"Ablation complete: {len(ablation_df)} rows")
#
# # Re-run hypothesis tests with ablation data
# results = run_all_tests(MODEL_NAME)
# h5 = results.get('H5', {})
# if h5.get('passed'):
#     print(f"H5 (ablation sensitivity): PASSED")
# else:
#     print(f"H5: {h5}")